In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 15


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.5789029970765114
Epoch 2/100, Loss: 2.6586594730615616
Epoch 3/100, Loss: 2.714067041873932
Epoch 4/100, Loss: 2.5687753409147263
Epoch 5/100, Loss: 2.3362646475434303
Epoch 6/100, Loss: 2.8832333609461784
Epoch 7/100, Loss: 2.3314481042325497
Epoch 8/100, Loss: 2.9881894513964653
Epoch 9/100, Loss: 2.8491171449422836
Epoch 10/100, Loss: 2.390319839119911
Epoch 11/100, Loss: 2.622142069041729
Epoch 12/100, Loss: 2.738435171544552
Epoch 13/100, Loss: 2.5588227286934853
Epoch 14/100, Loss: 2.445123016834259
Epoch 15/100, Loss: 2.6755814030766487
Epoch 16/100, Loss: 2.5508252009749413
Epoch 17/100, Loss: 2.6369319409132004
Epoch 18/100, Loss: 2.3864798322319984


Epoch 19/100, Loss: 3.0649977922439575
Epoch 20/100, Loss: 2.420788623392582
Epoch 21/100, Loss: 2.664231337606907
Epoch 22/100, Loss: 2.615966349840164
Epoch 23/100, Loss: 2.7308124229311943
Epoch 24/100, Loss: 2.640966333448887
Epoch 25/100, Loss: 2.4419243782758713
Epoch 26/100, Loss: 2.3015841245651245
Epoch 27/100, Loss: 2.5224331840872765
Epoch 28/100, Loss: 2.3971783742308617
Epoch 29/100, Loss: 2.629357486963272
Epoch 30/100, Loss: 2.4893036633729935


Epoch 31/100, Loss: 2.5335932299494743
Epoch 32/100, Loss: 2.772405691444874
Epoch 33/100, Loss: 2.747854046523571
Epoch 34/100, Loss: 2.475302740931511
Epoch 35/100, Loss: 2.622665837407112
Epoch 36/100, Loss: 2.6804391369223595
Epoch 37/100, Loss: 2.5834754183888435
Epoch 38/100, Loss: 2.769499033689499
Epoch 39/100, Loss: 2.603186570107937
Epoch 40/100, Loss: 2.588120475411415
Epoch 41/100, Loss: 2.587360478937626
Epoch 42/100, Loss: 2.477099873125553
Epoch 43/100, Loss: 2.552118182182312
Epoch 44/100, Loss: 2.7188212797045708


Epoch 45/100, Loss: 2.6325052604079247
Epoch 46/100, Loss: 2.4601808711886406
Epoch 47/100, Loss: 2.604835532605648
Epoch 48/100, Loss: 2.648418106138706
Epoch 49/100, Loss: 2.724471129477024
Epoch 50/100, Loss: 2.596586264669895
Epoch 51/100, Loss: 2.502923868596554
Epoch 52/100, Loss: 2.446785405278206
Epoch 53/100, Loss: 2.5515106692910194
Epoch 54/100, Loss: 2.590499997138977
Epoch 55/100, Loss: 2.701975777745247
Epoch 56/100, Loss: 2.619287133216858


Epoch 57/100, Loss: 2.6640863195061684
Epoch 58/100, Loss: 2.5473972484469414
Epoch 59/100, Loss: 2.4409705474972725
Epoch 60/100, Loss: 3.035548411309719
Epoch 61/100, Loss: 2.493886820971966
Epoch 62/100, Loss: 2.518955811858177
Epoch 63/100, Loss: 3.2668636068701744
Epoch 64/100, Loss: 2.5041882023215294
Epoch 65/100, Loss: 2.683095544576645
Epoch 66/100, Loss: 2.700582079589367
Epoch 67/100, Loss: 2.6260511577129364
Epoch 68/100, Loss: 2.291816495358944


...

Epoch 48/100, Loss: 4.152287989854813
Epoch 49/100, Loss: 4.314691826701164
Epoch 50/100, Loss: 4.15459805727005
Epoch 51/100, Loss: 4.220604956150055
Epoch 52/100, Loss: 4.171986594796181
Epoch 53/100, Loss: 4.366397708654404
Epoch 54/100, Loss: 4.1750060468912125
Epoch 55/100, Loss: 4.16479130089283
Epoch 56/100, Loss: 4.123158738017082
Epoch 57/100, Loss: 4.325121626257896
Epoch 58/100, Loss: 4.07977519929409
Epoch 59/100, Loss: 4.365240827202797
Epoch 60/100, Loss: 4.211534008383751


Epoch 61/100, Loss: 4.114695996046066
Epoch 62/100, Loss: 4.208595126867294
Epoch 63/100, Loss: 4.214175537228584
Epoch 64/100, Loss: 4.16544497013092
Epoch 65/100, Loss: 4.197631552815437
Epoch 66/100, Loss: 4.232948139309883
Epoch 67/100, Loss: 4.0798221081495285
Epoch 68/100, Loss: 4.386285424232483
Epoch 69/100, Loss: 3.9993532299995422
Epoch 70/100, Loss: 4.230748504400253
Epoch 71/100, Loss: 4.271113455295563
Epoch 72/100, Loss: 4.1680581867694855


Epoch 73/100, Loss: 4.401632562279701
Epoch 74/100, Loss: 4.135488912463188
Epoch 75/100, Loss: 4.313650622963905
Epoch 76/100, Loss: 4.093475446105003
Epoch 77/100, Loss: 4.1605531722307205
Epoch 78/100, Loss: 4.250165045261383
Epoch 79/100, Loss: 4.196987837553024
Epoch 80/100, Loss: 4.250782534480095
Epoch 81/100, Loss: 4.063557341694832
Epoch 82/100, Loss: 4.6746761947870255
Epoch 83/100, Loss: 4.380501076579094
Epoch 84/100, Loss: 4.21792796254158


Epoch 85/100, Loss: 4.138033777475357
Epoch 86/100, Loss: 4.254954665899277
Epoch 87/100, Loss: 4.1587091982364655
Epoch 88/100, Loss: 4.131085827946663
Epoch 89/100, Loss: 4.224187925457954
Epoch 90/100, Loss: 4.3412803411483765
Epoch 91/100, Loss: 4.17736092209816
Epoch 92/100, Loss: 4.134124547243118
Epoch 93/100, Loss: 4.324079245328903
Epoch 94/100, Loss: 4.17545747756958


Epoch 95/100, Loss: 4.309278905391693
Epoch 96/100, Loss: 4.382511928677559
Epoch 97/100, Loss: 4.231472909450531
Epoch 98/100, Loss: 4.271222323179245
Epoch 99/100, Loss: 4.538745433092117
Epoch 100/100, Loss: 4.367295816540718
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5743
